In [4]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("B:/Projects/Disertatie/Diabetic-Retinopathy-Classifier"),
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "datasets").exists() and (candidate / "Latex").exists():
            return candidate
    raise FileNotFoundError("Nu am gasit radacina proiectului Diabetic-Retinopathy-Classifier.")


PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / "Notebooks" / "Rezultate" / "New" / "grafice" / "distributii_date_train" / "distributie_train_dataseturi.csv"
OUTPUT_DIR = CSV_PATH.parent
LATEX_FIGURES_DIR = PROJECT_ROOT / "Latex" / "Figuri"

CLASS_COLUMNS = ["Clasa 0", "Clasa 1", "Clasa 2", "Clasa 3", "Clasa 4"]
COLORS = ["#2f6fbb", "#4aa96c", "#f2b84b", "#d96b43", "#8a5cc2"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DATASETS = {
    "APTOS": {
        "title": "APTOS 2019",
        "output": "distributie_train_aptos.png",
        "split_folders": PROJECT_ROOT / "datasets" / "processed_by_me" / "aptos" / "aptos_augmented",
        "csv_splits": {
            "train": PROJECT_ROOT / "datasets" / "originals" / "aptos_normal" / "train_1.csv",
            "val": PROJECT_ROOT / "datasets" / "originals" / "aptos_normal" / "valid.csv",
            "test": PROJECT_ROOT / "datasets" / "originals" / "aptos_normal" / "test.csv",
        },
    },
    "EyePACS": {
        "title": "EyePACS",
        "output": "distributie_train_eyepacs.png",
        "split_folders": PROJECT_ROOT / "datasets" / "processed_by_me" / "eyepacs" / "eyepacs_augmented",
        "csv_splits": {
            "train": PROJECT_ROOT / "datasets" / "originals" / "EyePACS" / "original_train_labels.csv",
            "test": PROJECT_ROOT / "datasets" / "originals" / "EyePACS" / "original_test_labels.csv",
        },
    },
    "Balanced_Aug": {
        "title": "Balanced_Aug",
        "output": "distributie_train_balanced_aug.png",
        "split_folders": PROJECT_ROOT / "datasets" / "processed_by_me" / "balanced_aug" / "balanced_augmented",
        "csv_splits": {
            "train": PROJECT_ROOT / "datasets" / "originals" / "Diabetic_Balanced_Aug" / "trainLabels.csv",
        },
    },
}


def count_csv_rows(path):
    if path is None or not path.exists():
        return None
    return len(pd.read_csv(path))


def count_images(path):
    if not path.exists():
        return None
    return sum(
        1
        for image_path in path.rglob("*")
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
    )


def format_count(value):
    if value is None:
        return "nu exista"
    return f"{value:,}".replace(",", ".")


def print_split_counts():
    print("Numar imagini pe split")
    for dataset_name, meta in DATASETS.items():
        print(f"\n{dataset_name}:")
        split_root = meta["split_folders"]
        csv_splits = meta.get("csv_splits", {})
        for split in ["train", "val", "test"]:
            csv_count = count_csv_rows(csv_splits.get(split))
            folder_count = count_images(split_root / split)
            if csv_count is not None:
                print(f"  {split}: {format_count(csv_count)} imagini in CSV; {format_count(folder_count)} imagini in folderul procesat")
            else:
                print(f"  {split}: CSV indisponibil; {format_count(folder_count)} imagini in folderul procesat")


def plot_dataset(dataset_name, row, meta):
    values = row[CLASS_COLUMNS].astype(int)

    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=180)
    bars = ax.bar(CLASS_COLUMNS, values, color=COLORS, edgecolor="#1f2933", linewidth=0.8)

    ax.set_title(f"Distributia imaginilor de antrenare - {meta['title']}", fontsize=15, weight="bold")
    ax.set_xlabel("Clasa de retinopatie diabetica")
    ax.set_ylabel("Numar imagini")
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
    ax.set_axisbelow(True)

    ymax = max(values) * 1.16
    ax.set_ylim(0, ymax)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + ymax * 0.015,
            format_count(int(value)),
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()

    output_path = OUTPUT_DIR / meta["output"]
    latex_output_path = LATEX_FIGURES_DIR / meta["output"]
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)
    shutil.copy2(output_path, latex_output_path)
    return output_path, latex_output_path


print_split_counts()

df = pd.read_csv(CSV_PATH, index_col=0)
print("\nGrafice generate:")
for dataset_name, meta in DATASETS.items():
    output_path, latex_output_path = plot_dataset(dataset_name, df.loc[dataset_name], meta)
    print(f"  {dataset_name}: {output_path}")
    print(f"  copie LaTeX: {latex_output_path}")


Numar imagini pe split

APTOS:
  train: 2.930 imagini in CSV; 6.552 imagini in folderul procesat
  val: 366 imagini in CSV; 733 imagini in folderul procesat
  test: 366 imagini in CSV; 367 imagini in folderul procesat

EyePACS:
  train: 35.126 imagini in CSV; 64.530 imagini in folderul procesat
  val: CSV indisponibil; 5.506 imagini in folderul procesat
  test: 53.576 imagini in CSV; 2.753 imagini in folderul procesat

Balanced_Aug:
  train: 35.126 imagini in CSV; 33.691 imagini in folderul procesat
  val: CSV indisponibil; 4.055 imagini in folderul procesat
  test: CSV indisponibil; 2.028 imagini in folderul procesat

Grafice generate:
  APTOS: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\Notebooks\Rezultate\New\grafice\distributii_date_train\distributie_train_aptos.png
  copie LaTeX: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\Latex\Figuri\distributie_train_aptos.png
  EyePACS: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\Notebooks\Rezultate\New\grafice